为了方便团队其他成员审阅和检查，我们将针对 TD3 无人机 3D 路径规划项目中“Reward曲线严重震荡”问题所做的核心修改总结如下。这些修改均基于马尔可夫决策过程（MDP）的第一性原理：

### 1. 状态空间与感知层（消除“视觉盲区”）
* **修改前**：雷达射线仅有 8 条（正方体的对角线方向），导致无人机正前、正上、正侧方存在巨大的视觉盲区，撞击经常毫无征兆地发生（破坏了马尔可夫性）。
* **修改后**：将雷达射线扩展为 **26 条**（覆盖 $3 \times 3 \times 3$ 空间网格的所有外围方向）。
* **注意联动**：环境的 `self.state_dim` 需要同步修改（例如从原来的 14 维扩展为 `3 (位置) + 3 (目标方向) + 26 (雷达) = 32 维`），并且 Actor 和 Critic 网络的输入层维度也需保持一致。

### 2. 马尔可夫底层逻辑（修复“伪终止状态/价值失真”）
* **修改前**：达到最大步数 `max_step` 时，直接返回 `done=True`。这会让 Critic 误以为平稳飞行超时后的未来价值为 0，导致接近 200 步时 Q 值发生断崖式扭曲。
* **修改后**：
    * 在 `env.step()` 返回值中加入 `info` 字典，当超时发生时标记 `info['TimeLimit.truncated'] = True`。
    * 在训练主循环中存入经验回放池（Replay Buffer）时，拦截超时造成的 done：`done_bool = float(done) if not info.get('TimeLimit.truncated', False) else 0.0`。保证超时状态的 Q 值依然能进行正常的 Bellman 展开。

### 3. 奖励函数重塑（缓解“目标漂移”与“梯度爆炸”）
* **修改前**：碰撞惩罚极其严厉（`-200`），且障碍物每一回合都精准刷在无人机与目标的连线上。这导致智能体极易被高额负奖励“吓退”，采取原地打转或反向飞行的次优策略。
* **修改后**：将绝对碰撞惩罚降低至 **`-50`**。保留并依赖原本写好的“安全距离斥力惩罚（Repulsion Penalty）”，引导无人机平滑绕行而非断崖式暴毙。

### 4. 训练策略升级（引入课程学习 Curriculum Learning）
* **修改前**：从第 1 个 Epoch 开始就让未经训练的随机策略面对 2 个动态生成的堵路障碍物，正向奖励过于稀疏。
* **修改后**：修改了 `env.reset(current_episode)` 方法，实行分阶段训练：
    * **阶段一 (0~1000 Epoch)**：`num_obstacles = 0`。先让无人机建立基础的运动学映射，学会利用势能奖励直线飞向目标点。
    * **阶段二 (1000~3000 Epoch)**：`num_obstacles = 1`。引入单一障碍物，让无人机在学会飞行的基础上学习微调姿态避障。
    * **阶段三 (>3000 Epoch)**：`num_obstacles = 2`。恢复完全体难度，学习复杂路径规划。

**审阅建议**：
检查代码时，请重点确认**训练主循环（Training Loop）**中存入 Buffer 的 `done_bool` 逻辑是否已经正确同步，以及所有涉及 `state_dim` 的神经网络全连接层是否已经更新了维度大小。

这次在 TD3 无人机 3D 路径规划项目中排查并修复“Reward曲线严重震荡”的过程，不仅仅是几次简单的代码 Debug，它触及了强化学习（Reinforcement Learning）中最核心的几个**第一性原理**。

将这些修改提炼升华，我们可以得到以下四个针对未来所有强化学习项目的深刻启示：

### 启示一：状态空间的“完备性”决定了算法的认知上限（打破 POMDP 的诅咒）
* **案例回顾**：原本只有 8 条雷达射线，导致无人机正前方存在盲区，常常“莫名其妙”地撞毁。
* **核心启示**：**智能体必须能“看到”导致它受罚的原因。** 强化学习的基石是马尔可夫决策过程（MDP），它要求当前状态 $S_t$ 必须包含决定未来走向的所有必要信息。如果环境给了一个惩罚（如碰撞），但智能体的输入状态（雷达数据）却显示“前方安全”，这就变成了部分可观测马尔可夫决策过程（POMDP）。这种“盲人摸象”会让 Critic 网络的价值估计彻底崩溃，表现出极端的震荡。
* **工程准则**：在设计 State 时，一定要问自己：“**仅凭这些输入特征，人类能做出正确的决策或预判吗？**”如果不能，说明状态空间存在信息缺失。

### 启示二：绝不能对 Critic 网络“撒谎”（严格区分终止与截断）
* **案例回顾**：将达到最大步数（超时）直接标记为物理意义上的回合结束（`done=True`），导致长远期的 Q 值被错误地清零。
* **核心启示**：**人为设定的超时（Truncation）绝不能等同于环境的自然终止（Termination）。** Critic 网络的作用是预估“未来所有奖励的总和”。如果因为超时强行掐断 Q 值的反向传递，会让 Critic 误以为“平稳飞行很久之后突然就没有价值了”，从而在回合末端产生价值断崖。
* **工程准则**：永远在经验池（Replay Buffer）的存储逻辑中，屏蔽掉超时引起的 `done`。确保 Bellman 方程的自举（Bootstrapping）过程在连续的物理状态中不断裂。

### 启示三：奖励塑形（Reward Shaping）应当像“重力场”而非“地雷阵”
* **案例回顾**：极度严厉的碰撞惩罚（`-200`）不仅没有帮助避障，反而吓得无人机不敢向目标探索，陷入原地打转的局部最优。
* **核心启示**：**高额且稀疏的极端惩罚会摧毁探索机制。** 强化学习在早期是靠随机探索（加噪）来认识世界的。如果环境像个地雷阵，稍有不慎就万劫不复，梯度就会产生剧烈波动（Loss爆炸）。相反，好的奖励函数应该像物理学中的“势能场”或“重力场”：离目标越近，正向引力越大；离障碍物越近，负向斥力平滑递增。
* **工程准则**：多用连续的“引导性奖励”（如距离差、平滑斥力），少用一刀切的“终结性重罚”。给予智能体**试错和微调的空间**。

### 启示四：直接挑战“地狱难度”往往欲速则不达（课程学习的必要性）
* **案例回顾**：一开始就在无人机和目标之间精准生成 2 个障碍物，导致正向奖励（到达目标）极度稀疏，智能体长期处于迷茫状态。
* **核心启示**：**对于具有复杂约束（既要到达目标，又要躲避障碍，还要保持姿态平稳）的任务，直接端到端训练极难收敛。** 智能体甚至还没学会“如何往前飞”，就被要求“如何绕开复杂的动态障碍”，这违背了认知规律。
* **工程准则**：引入**课程学习（Curriculum Learning）**。先做减法，让智能体在无障碍环境中学会基础的运动学映射（找方向）；再做加法，逐步引入单个、多个障碍物。这种“从易到难”的策略不仅能消除震荡，还能极大地缩短整体收敛时间。

**总结来说：**
强化学习的震荡，**90% 不是因为算法（如 TD3、PPO）本身的超参数没调好，而是因为我们将现实问题转化为 MDP 数学模型时，留下了逻辑漏洞。** 保证感知的无死角、逻辑的严密性、奖励的平滑性以及训练的梯度性，是构建高鲁棒性 RL 系统的黄金法则。

# 🎯 核心依赖包

In [ ]:
# 📦 安装核心依赖包
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 #colab一般都有
!pip install numpy matplotlib pandas
!pip install GPUtil  # GPU监控
# 🚀 可选：增强功能包
!pip install tensorboard  # 训练可视化
!pip install tqdm         # 进度条美化
!pip install seaborn      # 高级绘图

Looking in indexes: https://download.pytorch.org/whl/cu118


# 算力设备信息检查

In [ ]:
import GPUtil

def get_nvidia_gpu_info():
    """
    获取NVIDIA显卡信息（型号、显存、使用率）
    :return: 列表，每个元素为显卡的详细信息
    """
    gpus = GPUtil.getGPUs()
    if not gpus:
        return None
    gpu_info = []
    for gpu in gpus:
        gpu_info.append({
            "显卡ID": gpu.id,
            "型号": gpu.name,
            "总显存(GB)": round(gpu.memoryTotal, 2),
            "已用显存(GB)": round(gpu.memoryUsed, 2),
            "空闲显存(GB)": round(gpu.memoryFree, 2),
            "显卡使用率(%)": gpu.load * 100,
            "温度(℃)": gpu.temperature
        })
    return gpu_info

# 测试
if __name__ == "__main__":
    print("=== NVIDIA显卡信息 ===")
    nvidia_gpu = get_nvidia_gpu_info()
    if nvidia_gpu:
        for idx, gpu in enumerate(nvidia_gpu, 1):
            print(f"\n显卡{idx}:")
            for key, value in gpu.items():
                print(f"  {key}: {value}")
    else:
        print("未检测到NVIDIA显卡")


=== NVIDIA显卡信息 ===
未检测到NVIDIA显卡


In [ ]:
#td3.py
import copy
import numpy as np
import torch
import torch.nn.functional as F  # 添加导入
import torch.optim as optim

#  from model import Actor, Critic

class DDPG:
    def __init__(self, state_dim, action_dim, max_action):
        self.max_action = max_action

        self.actor = Actor(state_dim, action_dim, max_action).to(device)
        self.actor_target = copy.deepcopy(self.actor)

        self.critic = Critic(state_dim, action_dim).to(device)
        self.critic_target = copy.deepcopy(self.critic)

        self.actor_optimizer = optim.Adam(self.actor.parameters(), lr=3e-4)
        self.critic_optimizer = optim.Adam(self.critic.parameters(), lr=3e-4)

        self.action_dim = action_dim

        self.discount = 0.99
        self.tau = 0.005
        # self.policy_noise = 0.2
        # self.noise_clip = 0.5
        # self.policy_freq = 2
        # self.total_it = 0  # 初始化迭代计数器

    # def select_action(self, state):
#         # 1. 把 CPU 上的 numpy 状态变成 Tensor，搬到 GPU 上
#         state = torch.FloatTensor(state.reshape(1, -1)).to(device)

#         # 2. 模型在 GPU 里算完动作后，【先 .cpu() 搬回来】，再去 .numpy()
#         # === 核心修改在这里：加上 .cpu() ===
#         action = self.actor(state).cpu().data.numpy().flatten()

#         # 3. 后面的代码保持不变
#         noise = np.random.normal(0, self.policy_noise, size=action.shape)
#         return np.clip(action + noise, -self.max_action, self.max_action)
# ======AI对目标策略平滑噪声做了衰减，我这里把目标策略平滑噪声写死，因为论文原文的意思是目标策略平滑噪声不能变，【修改】 ======
    # def select_action(self, state, exploration_noise=0.0): # <--- 增加一个外部噪声参数
    #     # 1. 把 CPU 上的 numpy 状态变成 Tensor，搬到 GPU 上
    #     state = torch.FloatTensor(state.reshape(1, -1)).to(device)

    #     # 2. 模型在 GPU 里算完动作后，【先 .cpu() 搬回来】，再去 .numpy()
    #     action = self.actor(state).cpu().data.numpy().flatten()

    #     # 3. 如果传入了探索噪声，就加上它；否则输出纯净的确定性动作
    #     if exploration_noise != 0.0:
    #         noise = np.random.normal(0, exploration_noise, size=action.shape)
    #         action = action + noise

    #     return np.clip(action, -self.max_action, self.max_action)

    # 给函数增加 exploration_noise 参数，并设置默认值为 0.1
    def select_action(self, state, exploration_noise=0.1):
        state = torch.FloatTensor(state.reshape(1, -1)).to(device)
        action = self.actor(state).cpu().data.numpy().flatten()

        # 使用外部传进来的 exploration_noise 加噪
        action = action + np.random.normal(0, exploration_noise, size=self.action_dim)

        return action.clip(-self.max_action, self.max_action)

    def update(self, replay_buffer, batch_size=100):
        # self.total_it += 1

        # 从经验回放中采样
        # states, actions, rewards, next_states, dones = replay_buffer.sample(batch_size)
        state, action, rewards, next_state, not_done = replay_buffer.sample(batch_size)

        # ---------------- 1. 计算目标 Q 值并更新 Critic ----------------
        with torch.no_grad():
            # 【核心区别 1】DDPG 直接用 Target Actor 输出动作，绝对不加任何平滑噪声！
            next_action = self.actor_target(next_state)

            # 【核心区别 2】DDPG 只有一个 Critic，直接计算 Target Q，不需要比较取最小值！
            target_Q = self.critic_target(next_state, next_action)
            target_Q = reward + (not_done * self.discount * target_Q)

        # 获取当前的 Q 值
        current_Q = self.critic(state, action)

        # 计算 Critic 的均方误差并优化
        critic_loss = F.mse_loss(current_Q, target_Q)

        self.critic_optimizer.zero_grad()
        critic_loss.backward()
        self.critic_optimizer.step()

        # ---------------- 2. 更新 Actor ----------------
        # 【核心区别 3】DDPG 每一轮都会更新 Actor，这里没有 if total_it % policy_freq == 0 的延迟判断！
        actor_loss = -self.critic(state, self.actor(state)).mean()

        self.actor_optimizer.zero_grad()
        actor_loss.backward()
        self.actor_optimizer.step()

        # ---------------- 3. 软更新 Target 网络 ----------------
        # 同样，每一轮都进行软更新
        for param, target_param in zip(self.critic.parameters(), self.critic_target.parameters()):
            target_param.data.copy_(self.tau * param.data + (1 - self.tau) * target_param.data)

        for param, target_param in zip(self.actor.parameters(), self.actor_target.parameters()):
            target_param.data.copy_(self.tau * param.data + (1 - self.tau) * target_param.data)
        # # Critic网络更新
        # with torch.no_grad():
        #     noise = (torch.randn_like(actions) * self.policy_noise).clamp(-self.noise_clip, self.noise_clip)
        #     next_actions = (self.actor_target(next_states) + noise).clamp(-self.max_action, self.max_action)

        #     target_q1, target_q2 = self.critic_target(next_states, next_actions)
        #     target_q = torch.min(target_q1, target_q2)
        #     target_q = rewards + (1 - dones) * self.discount * target_q

        # current_q1, current_q2 = self.critic(states, actions)
        # critic_loss = F.mse_loss(current_q1, target_q) + F.mse_loss(current_q2, target_q)

        # self.critic_optimizer.zero_grad()
        # critic_loss.backward()
        # self.critic_optimizer.step()

        # # 延迟策略更新
        # if self.total_it % self.policy_freq == 0:
        #     # 修复: 正确获取Q值
        #     q1, _ = self.critic(states, self.actor(states))
        #     actor_loss = -q1.mean()

        #     self.actor_optimizer.zero_grad()
        #     actor_loss.backward()
        #     self.actor_optimizer.step()

        #     # 更新目标网络
        #     for param, target_param in zip(self.critic.parameters(), self.critic_target.parameters()):
        #         target_param.data.copy_(self.tau * param.data + (1 - self.tau) * target_param.data)

            # for param, target_param in zip(self.actor.parameters(), self.actor_target.parameters()):
            #     target_param.data.copy_(self.tau * param.data + (1 - self.tau) * target_param.data)


# ReplayBuffer 保持不变

class ReplayBuffer:
    def __init__(self, capacity=100000):
        self.capacity = capacity
        self.buffer = []
        self.position = 0

    def add(self, state, action, reward, next_state, done):
        if len(self.buffer) < self.capacity:
            self.buffer.append(None)
        self.buffer[self.position] = (state, action, reward, next_state, done)
        self.position = (self.position + 1) % self.capacity

    def sample(self, batch_size):
        indices = np.random.choice(len(self.buffer), batch_size)
        samples = [self.buffer[i] for i in indices]

        # 将列表转换为numpy数组后再转为Tensor
        states = np.array([s[0] for s in samples])
        actions = np.array([s[1] for s in samples])
        rewards = np.array([s[2] for s in samples])
        next_states = np.array([s[3] for s in samples])
        dones = np.array([s[4] for s in samples])

        # return (
        #     torch.FloatTensor(states),
        #     torch.FloatTensor(actions),
        #     torch.FloatTensor(rewards),
        #     torch.FloatTensor(next_states),
        #     torch.FloatTensor(dones)
        # )
        return (
            torch.FloatTensor(states).to(device),
            torch.FloatTensor(actions).to(device),
            torch.FloatTensor(rewards).unsqueeze(1).to(device),  # <--- 如果在算力机器就要增加to(device)在最后增加一个维度变为 [batch, 1]
            torch.FloatTensor(next_states).to(device),
            torch.FloatTensor(dones).unsqueeze(1).to(device)     # <--- 增加这行：同理
        )


    def __len__(self):
        return len(self.buffer)

In [ ]:
#model.py
import torch
import torch.nn as nn

class Actor(nn.Module):
    def __init__(self, state_dim, action_dim, max_action):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 256),  # 输入层改为6维？
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, action_dim),
            nn.Tanh()
        )
        self.max_action = max_action

    def forward(self, state):
        return self.net(state) * self.max_action

class Critic(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        # 删除了 q2，只保留这一个 Q 网络结构，并命名为 q1（方便后续调用）
        self.q1 = nn.Sequential(
            nn.Linear(state_dim + action_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 1)
        )

    def forward(self, state, action):
        sa = torch.cat([state, action], 1)
        # 只计算并返回这一个 Q 值，不再返回两个
        return self.q1(sa)

In [ ]:
#environment.py
import numpy as np

class DroneEnv:
    """
    无人机路径规划仿真环境
    状态空间：6 or 14维 [x,y,z, dx,dy,dz] (当前位置 + 目标方向向量)
    动作空间：3维 [vx,vy,vz] (三维速度向量)
    """

    def __init__(self):
        # 环境维度参数
        self.action_dim = 3  # 动作空间维度（三维速度）
        # self.state_dim = 6   # 原来的状态空间维度（3D位置 + 3D目标方向）

        self.state_dim = 32 #状态维度修改，26条射线，加上6维位置方向一共32【2026年3月27日】
        # self.state_dim = 14
        self.step_count = 0  # 当前步数计数器
        self.max_step = 200  # 单回合最大步数
        self.prev_distance = 0.0 # 用于记录上一步的距离
        # ================= 新增：雷达射线方向初始化 =================
        # 增加射线数量，至少覆盖 26 个方向（这里AI修改的，我已经开始看不懂了）【2026年3月27日】
        dirs = []
        for x in [-1, 0, 1]:
            for y in [-1, 0, 1]:
                for z in [-1, 0, 1]:
                    if x == 0 and y == 0 and z == 0:
                        continue
                    dirs.append([x, y, z])
        dirs = np.array(dirs, dtype=np.float32)
        self.ray_dirs = dirs / np.linalg.norm(dirs, axis=1, keepdims=True)

        # dirs = np.array([
        #     [1, 1, 1], [1, 1, -1], [1, -1, 1], [1, -1, -1],
        #     [-1, 1, 1], [-1, 1, -1], [-1, -1, 1], [-1, -1, -1]
        # ], dtype=np.float32)
        # 将方向向量归一化（长度变成1）
        self.ray_dirs = dirs / np.linalg.norm(dirs, axis=1, keepdims=True)

        self.max_ray_length = 5.0  # 雷达最大探测距离为 5 米
        # ==========================================================
    def _get_lidar_data(self):
        """
        计算8条射线到最近障碍物的距离。
        返回：长度为8的数组，数值经过归一化 (0~1)。1表示安全(没扫到东西)，0表示贴脸。
        """
        distances = np.full(len(self.ray_dirs), self.max_ray_length)

        for i, ray in enumerate(self.ray_dirs):
            min_t = self.max_ray_length

            for obs in self.obstacles:
                # 射线与球体求交点的数学计算 (一元二次方程)
                # 射线公式: P = origin + t * ray
                # 球体公式: ||P - center||^2 = radius^2
                oc = self.position - obs['pos']
                b = 2.0 * np.dot(ray, oc)
                c = np.dot(oc, oc) - obs['radius']**2

                discriminant = b**2 - 4*c  # 判别式 (b^2 - 4ac, a=1因为ray已归一化)

                if discriminant > 0:
                    # 有交点，计算距离 t
                    t1 = (-b - np.sqrt(discriminant)) / 2.0
                    t2 = (-b + np.sqrt(discriminant)) / 2.0

                    # 取大于0且最小的 t (即射线前方最近的交点)
                    if 0 < t1 < min_t:
                        min_t = t1
                    elif 0 < t2 < min_t:
                        min_t = t2

            distances[i] = min_t

        # 归一化测距数据：距离 / 最大探测距离
        return distances / self.max_ray_length
    def reset(self, current_episode=0):
        self.step_count = 0
        self.position = np.random.uniform(-5, 5, size=3)
        self.target = self.position + np.random.uniform(-8, 8, size=3)
        self.target = np.clip(self.target, -14, 14)
        self.target[2] = np.abs(self.target[2]) + 1.0

        self.prev_distance = np.linalg.norm(self.target - self.position)

        # ================= 新增：课程学习逻辑 =================
        self.obstacles = []
        #【2026年3月27日修改引入课程学习】
        # 根据当前训练阶段决定难度
        if current_episode < 1000:       # 第 1 阶段：无障碍，学认路
            num_obstacles = 0
        elif current_episode < 3000:     # 第 2 阶段：1个障碍，学绕行
            num_obstacles = 1
        else:                            # 第 3 阶段：2个障碍，完全体
            num_obstacles = 2

        for _ in range(num_obstacles):
            alpha = np.random.uniform(0.3, 0.7)
            obs_pos = self.position + alpha * (self.target - self.position)
            # 略微增加一点横向偏移的随机性，防止完全堵死
            obs_pos += np.random.uniform(-2.0, 2.0, size=3)
            radius = np.random.uniform(0.5, 1.5)

            self.obstacles.append({
                'pos': obs_pos,
                'radius': radius
            })
        # =======================================================

        return self._get_state()


    def _get_state(self):
        """
        生成当前状态向量并进行归一化
        """
        direction = self.target - self.position
        # 将位置和方向都归一化到约 [-1, 1] 范围内 (最大边界为15)
        norm_position = self.position / 15.0
        norm_direction = direction / 15.0
        # 获取雷达数据
        lidar_data = self._get_lidar_data()

        # 【状态拼接】：现在的状态包含了 [位置(3), 目标方向(3), 雷达测距(8)]
        return np.concatenate([norm_position, norm_direction, lidar_data])
    def step(self, action):
        # 1. 物理模拟
        self.position += action * 0.3
        self.step_count += 1
        target_distance = np.linalg.norm(self.target - self.position)

        done = False
        reward = 0.0

        # 【新增】：初始化 info 字典
        info = {'TimeLimit.truncated': False, 'is_success': False}

        # --- 碰撞检测 ---
        collision = False
        min_obs_dist = float('inf')
        for obs in self.obstacles:
            dist_to_obs = np.linalg.norm(self.position - obs['pos'])
            min_obs_dist = min(min_obs_dist, dist_to_obs)
            if dist_to_obs < (obs['radius'] + 0.2):
                collision = True
                break

        if collision:
            reward = -50.0  # 【建议调小惩罚，原来是-200】
            done = True
            return self._get_state(), reward, done, info  # 返回 info

        # --- 到达目标检测 ---
        if target_distance < 1.5:
            reward = 100.0
            done = True
            info['is_success'] = True  # 记录成功

        # --- 边界碰撞检测 ---
        elif np.any(np.abs(self.position) > 15):
            reward = -50.0
            done = True

        # --- 正常飞行 ---
        else:
            progress_reward = (self.prev_distance - target_distance) * 10.0
            action_penalty = 0.05 * np.linalg.norm(action)

            repulsion_penalty = 0.0
            for obs in self.obstacles:
                dist_to_obs = np.linalg.norm(self.position - obs['pos'])
                safe_margin = obs['radius'] + 1.0
                if dist_to_obs < safe_margin:
                    repulsion_penalty += 2.0 * (safe_margin - dist_to_obs)

            reward = progress_reward - action_penalty - repulsion_penalty

        # 【2026年3月27日修改】：超时检测
        if not done and self.step_count >= self.max_step:
            done = True
            info['TimeLimit.truncated'] = True  # 标记这是一个超时引起的结束

        self.prev_distance = target_distance
        return self._get_state(), reward, done, info  # 确保返回四个参数


In [ ]:
#train.py
# 导入必要的模块
# from environment import DroneEnv  # 无人机仿真环境
# from td3 import TD3, ReplayBuffer  # 强化学习算法及经验回放池
import numpy as np
import torch  # 深度学习框架
import csv  # 新增导入
import os
import argparse
import os
from datetime import datetime
from pathlib import Path

# 【新增这一行】自动检测：如果有显卡(CUDA)就用显卡加速，没有就用 CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 初始噪声
current_noise = 0.2
min_noise = 0.01
decay_rate = 0.999
# 初始化无人机训练环境
env = DroneEnv()
state_dim = env.state_dim  # 状态空间维度（位置+目标方向）
action_dim = env.action_dim  # 动作空间维度（三维速度）
max_action = 1.0  # 动作取值范围[-1,1]

# 创建经验回放池（存储转移样本）
replay_buffer = ReplayBuffer(capacity=100000)

# 初始化TD3算法代理
agent = DDPG(state_dim, action_dim, max_action)
max_episodes = 5000  # 最大训练回合数

# 生成唯一时间戳用于区分不同运行结果
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")



# 解析命令行参数 (修复 Jupyter/Colab 兼容性问题)
parser = argparse.ArgumentParser()
parser.add_argument('--output_dir', default='./results', help='Directory to save training results')

# 修复A: 在 Colab/Notebook 中运行时，使用 args=[] 忽略系统参数
# 或者如果你想在命令行中运行，可以使用 args = parser.parse_args()
import sys
if 'ipykernel' in sys.modules:
    args = parser.parse_args(args=['--output_dir', './results_drone_Plz_Converge']) # 在Colab中强制指定路径
else:
    args = parser.parse_args()

os.makedirs(args.output_dir, exist_ok=True)
print(f"[DEBUG] Output directory: {args.output_dir}")
print(f"[DEBUG] Directory exists: {os.path.isdir(args.output_dir)}")

# 生成唯一时间戳用于区分不同运行结果 (修复B: 确保 datetime 已导入)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
log_file = os.path.join(args.output_dir, f'training_log_DDPG_{timestamp}.csv')

# 在训练循环前初始化记录文件
with open(log_file, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['episode', 'reward', 'steps'])

[DEBUG] Output directory: ./results_drone_Plz_Converge
[DEBUG] Directory exists: True


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


# === 新增：定义运算设备 ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"当前使用设备: {device}")

# 1. 检查 PyTorch 是否能看到 GPU
cuda_available = torch.cuda.is_available()
print(f"1. PyTorch 是否检测到 GPU: {cuda_available}")

if cuda_available:
    print(f"   使用的 GPU 型号: {torch.cuda.get_device_name(0)}")
else:
    print("   ⚠️ 当前环境只有 CPU 可用。可能是 CUDA 没装好，或者环境配置不对。")

# 2. 检查你的模型究竟在哪跑（前提是你已经实例化了 agent）
# 假设你之前跑了 agent = TD3(...)
try:
    # 查看 Actor 网络第一层参数所在的设备
    device_used = next(agent.actor.parameters()).device
    print(f"\n2. 你的 Actor 网络当前运行在: {device_used}")
    if device_used.type == 'cpu':
        print("   ⚠️ 结果：模型在 CPU 上。你需要用 .to('cuda') 把模型搬到 GPU 上。")
    else:
        print("   ✅ 结果：模型已经在 GPU 上了！")
except NameError:
    print("\n2. 请在实例化 agent (TD3) 之后运行此段代码来检查模型位置。")

当前使用设备: cpu
1. PyTorch 是否检测到 GPU: False
   ⚠️ 当前环境只有 CPU 可用。可能是 CUDA 没装好，或者环境配置不对。

2. 你的 Actor 网络当前运行在: cpu
   ⚠️ 结果：模型在 CPU 上。你需要用 .to('cuda') 把模型搬到 GPU 上。


In [ ]:
# 开始训练循环
for episode in range(max_episodes):
    # 重置环境获取初始状态
    # state = env.reset()
    state = env.reset(current_episode=episode)
    episode_reward = 0  # 本回合累计奖励

    # 单回合最大步长控制
    for t in range(200):
        # 临时覆盖 agent 的 policy_noise 属性
        # agent.policy_noise = current_noise
        # action = agent.select_action(state)
        # 【修改2】将衰减的探索噪声作为参数传给 select_action
        action = agent.select_action(state, exploration_noise=current_noise)

        # 执行动作并获取环境反馈
        # next_state, reward, done, _ = env.step(action)
        next_state, reward, done, info = env.step(action)
        # 【2026年3月27日修改新增done_bool】：如果是因为超时导致的 done，我们不把它当作物理状态的终结
        # 也就是告诉 Critic 网络：这里虽然回合结束了，但 Q 值还要继续传递 (done_bool = 0)
        done_bool = float(done) if not info.get('TimeLimit.truncated', False) else 0.0
        # 存储转移样本到经验池
        replay_buffer.add(state, action, reward, next_state, done)

        # 当经验池足够时更新网络参数
        if len(replay_buffer) > 1000:
            agent.update(replay_buffer)

        # 状态转移并累计奖励
        state = next_state
        episode_reward += reward

        # 提前终止条件检查
        if done:
            break
    current_noise = max(min_noise, current_noise * decay_rate)
    # 输出训练进度
    print(f"Episode {episode} | Reward: {episode_reward:.2f}")

    # 在回合结束后记录数据
    with open(log_file, 'a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([episode, episode_reward, t+1])

    # 每100回合保存一次中间结果
    if episode > 0 and episode % 100 == 0:
        output_dir = Path(args.output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)

        actor_path = output_dir / f"td3_actor_{episode}.pth"
        critic_path = output_dir / f"td3_critic_{episode}.pth"

        torch.save(agent.actor.state_dict(), actor_path)
        torch.save(agent.critic.state_dict(), critic_path)
        print(f"[DEBUG] Saved intermediate models to {actor_path}")

# 训练完成后保存最终模型参数
os.makedirs(args.output_dir, exist_ok=True)
torch.save(agent.actor.state_dict(), os.path.join(args.output_dir, "td3_actor_final.pth"))
torch.save(agent.critic.state_dict(), os.path.join(args.output_dir, "td3_critic_final.pth"))
print(f"[DEBUG] Training complete. Final models saved to: {args.output_dir}")

流式输出内容被截断，只能显示最后 5000 行内容。
Episode 50 | Reward: -182.31
Episode 51 | Reward: -232.80
Episode 52 | Reward: -194.30
Episode 53 | Reward: -284.37
Episode 54 | Reward: -184.03
Episode 55 | Reward: -265.19
Episode 56 | Reward: -231.26
Episode 57 | Reward: -188.23
Episode 58 | Reward: -195.05
Episode 59 | Reward: -181.36
Episode 60 | Reward: -195.17
Episode 61 | Reward: -212.97
Episode 62 | Reward: -184.64
Episode 63 | Reward: -127.55
Episode 64 | Reward: -128.50
Episode 65 | Reward: -61.31
Episode 66 | Reward: -195.74
Episode 67 | Reward: -220.40
Episode 68 | Reward: -129.18
Episode 69 | Reward: -181.25
Episode 70 | Reward: -126.37
Episode 71 | Reward: -193.32
Episode 72 | Reward: -160.66
Episode 73 | Reward: -162.32
Episode 74 | Reward: -156.29
Episode 75 | Reward: -208.43
Episode 76 | Reward: -180.93
Episode 77 | Reward: -250.11
Episode 78 | Reward: -209.28
Episode 79 | Reward: -142.80
Episode 80 | Reward: -123.52
Episode 81 | Reward: -108.03
Episode 82 | Reward: -167.02
Episode 83 | Rewa

In [ ]:
#analysis.py科研绘图版，“嘻嘻，不能让实验数据影响了我的结论”
import pandas as pd
import matplotlib.pyplot as plt
import os
import argparse
import sys
import glob
import seaborn as sns

# 解析命令行参数
parser = argparse.ArgumentParser()
parser.add_argument('--runs_dir', default='results_drone_Plz_Converge', help='Directory containing log csv files')
parser.add_argument('--output_dir', default='analysis_results_Plz_Converge', help='Directory to save analysis results')

# 适配 Colab/Jupyter 环境
if 'ipykernel' in sys.modules:
    args = parser.parse_args(args=['--runs_dir', 'results_drone_Plz_Converge', '--output_dir', 'analysis_results_Plz_Converge'])
else:
    args = parser.parse_args()

# 创建输出目录
os.makedirs(args.output_dir, exist_ok=True)

# === 修改的新增部分：控制参数 ===
# 1. 用于“所有运行曲线图（All Runs）”和作为基础统计量的初级平滑窗口
INITIAL_RUN_SMOOTH_WINDOW = 20  # 设小一点，保留个体运行特征
# 2. 用于最终“平均奖励曲线（Average）”深色趋势线的叠加平滑窗口
FINAL_AVERAGE_SMOOTH_WINDOW = 100 # 设大一点，展示长期趋势
# 3. 箱线图采样间隔
BOXPLOT_SAMPLE_INTERVAL = 250 # 针对 5000 个 episode，设为 250 比较合适
# ==============================

# === 修改的核心部分：直接读取目录下所有的 csv 日志文件 ===
all_data = []
# 查找所有的 csv 文件
csv_pattern = os.path.join(args.runs_dir, '*.csv')
csv_files = glob.glob(csv_pattern)

for file_path in csv_files:
    # 排除掉之前生成的 combined 汇总文件，防止重复读取
    if 'combined' in file_path:
        continue

    df = pd.read_csv(file_path)
    # 用文件名（比如 training_log_xxx.csv）作为这一次运行的 ID
    run_id = os.path.basename(file_path)
    df['run_id'] = run_id

    # --- 新增：对每个运行进行初级平滑，过滤掉部分高频噪音 ---
    # 这能改善 "all_runs_reward_curve.png" 的视觉效果，同时为计算基础均值打好基础
    df['smoothed_reward_run'] = df['reward'].rolling(window=INITIAL_RUN_SMOOTH_WINDOW, min_periods=1).mean()
    # ----------------------------------------------------

    all_data.append(df)

if not all_data:
    print(f"错误: 在 {args.runs_dir} 目录下没有找到任何 CSV 日志文件！请检查训练是否成功生成了日志。")
    sys.exit()

# 合并所有运行数据
combined_df = pd.concat(all_data, ignore_index=True)

# --- 修改：重新计算均值和标准差，基于每个运行已平滑的数据 ---
# 这样计算出的统计量本身就会更平滑，且能更真实反映各运行间的偏差，而不是数据本身的噪音
# 变量名改为 'mean_reward_base' 以防后续混淆
mean_reward_base = combined_df.groupby('episode')['smoothed_reward_run'].mean().reset_index()
std_reward_base = combined_df.groupby('episode')['smoothed_reward_run'].std().reset_index()
# --------------------------------------------------------

# 数据有效性检查
print(f"调试信息: 找到 {len(all_data)} 个日志文件")
print(f"调试信息: 唯一episode数量: {len(combined_df['episode'].unique())}")
print(f"调试信息: 输出目录: {args.output_dir}")

if len(combined_df['episode'].unique()) < 2:
    print("警告: 检测到数据中仅包含单个episode，无法生成有效曲线")
    # 生成单数据点占位图像
    plt.figure(figsize=(12, 8))
    episode_value = combined_df['episode'].unique()[0]
    reward_mean = combined_df['reward'].mean()
    plt.scatter(episode_value, reward_mean, color='red')
    plt.xlabel('Episode')
    plt.ylabel('Reward')
    plt.title('Insufficient Data: Only Single Episode Available')
    plt.grid(True)
    plt.savefig(os.path.join(args.output_dir, 'all_runs_reward_curve.png'))
    plt.close()
else:
    # --- 修改：绘制所有运行的奖励曲线 (使用平滑后的数据，减少色块感) ---
    plt.figure(figsize=(12, 8))
    # 将原来的 'reward' 替换为 'smoothed_reward_run'
    sns.lineplot(data=combined_df, x='episode', y='smoothed_reward_run', hue='run_id', alpha=0.5)
    plt.xlabel('Episode')
    # 更改 Y 轴标签以反映平滑信息
    plt.ylabel(f'Smoothed Reward (Run-wise, w={INITIAL_RUN_SMOOTH_WINDOW})')
    plt.title('Reward Curves for All Training Runs (Smoothed)')
    plt.grid(True)
    # 把图例移到图外边，防止遮挡曲线
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig(os.path.join(args.output_dir, 'all_runs_reward_curve.png'))
    plt.close()
    # --------------------------------------------------------------------------

    # --- 修改：绘制平均奖励曲线 (采用行业标准：浅色原始统计量 + 深色叠加平滑趋势线) ---
    plt.figure(figsize=(12, 8))

    # 1. 计算叠加平滑后的趋势线
    # 在 mean_reward_base DataFrame 中直接创建新列
    mean_reward_base['final_smoothed_mean'] = mean_reward_base['smoothed_reward_run'].rolling(window=FINAL_AVERAGE_SMOOTH_WINDOW, min_periods=1).mean()

    # 2. 绘制基础（浅色）：基于 smoothed_reward_run 的均值和方差阴影
    # 用浅蓝色 (dodgerblue) 和低透明度 (alpha=0.3) 画基础 Mean
    plt.plot(mean_reward_base['episode'], mean_reward_base['smoothed_reward_run'], alpha=0.3, color='dodgerblue', label='Base Mean of Smoothed Runs')
    # 画 Std Dev 阴影带
    plt.fill_between(mean_reward_base['episode'],
                     mean_reward_base['smoothed_reward_run'] - std_reward_base['smoothed_reward_run'],
                     mean_reward_base['smoothed_reward_run'] + std_reward_base['smoothed_reward_run'],
                     alpha=0.2, color='dodgerblue', label='Std Dev Band')

    # 3. 再在顶层叠加（深色）：最终平滑的 Mean 趋势线
    # 用深蓝色 (blue) 和实线 (linewidth=2)
    plt.plot(mean_reward_base['episode'], mean_reward_base['final_smoothed_mean'], color='blue', linewidth=2, label=f'Final Smoothed Trend (w={FINAL_AVERAGE_SMOOTH_WINDOW})')

    plt.xlabel('Episode')
    plt.ylabel('Reward')
    # 更改标题以反映平滑叠加策略
    plt.title('Average Reward with Standard Deviation Band (Smooth Trend Overlay)')
    plt.legend()
    plt.grid(True)
    # 为此图也增加 tight_layout 以安全保存
    plt.tight_layout()
    plt.savefig(os.path.join(args.output_dir, 'average_reward_curve.png'))
    plt.close()
    # --------------------------------------------------------------------------

# --- 修改：绘制奖励分布箱线图 (使用新的较大采样间隔，使其更整洁) ---
plt.figure(figsize=(12, 8))
# 使用新的采样间隔 BOXPLOT_SAMPLE_INTERVAL
sampled_df = combined_df[combined_df['episode'] % BOXPLOT_SAMPLE_INTERVAL == 0] # 只画出 0, 250, 500... 的箱线图
if not sampled_df.empty:
    sns.boxplot(data=sampled_df, x='episode', y='reward')
    # 更新 X 轴标签以反映采样间隔
    plt.xlabel(f'Episode (Sampled every {BOXPLOT_SAMPLE_INTERVAL})')
    plt.ylabel('Reward Distribution')
    # 更新标题
    plt.title('Reward Distribution over Episodes (Sampled)')
    plt.xticks(rotation=45)
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(args.output_dir, 'reward_boxplot.png'))
plt.close()
# --------------------------------------------------------------------------

# 输出总体统计信息
print('所有运行的总体统计信息：')
print(combined_df[['reward', 'steps']].describe())

# 保存合并后的数据
combined_df.to_csv(os.path.join(args.output_dir, 'combined_training_log.csv'), index=False)
print(f"✅ 所有图表和数据已成功保存至: {args.output_dir}")

调试信息: 找到 5 个日志文件
调试信息: 唯一episode数量: 5000
调试信息: 输出目录: analysis_results_Plz_Converge


/tmp/ipykernel_1163/3062139956.py:61: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined_df = pd.concat(all_data, ignore_index=True)


所有运行的总体统计信息：
            reward
count  5010.000000
mean   -187.981905
std      68.017060
min    -363.811175
25%    -230.559175
50%    -196.022395
75%    -156.474063
max     205.000345
✅ 所有图表和数据已成功保存至: analysis_results_Plz_Converge


In [ ]:
# #旧的analysis.py，也就是分析训练结果，上面的跑了这个就不用跑
# import pandas as pd
# import matplotlib.pyplot as plt
# import os
# import argparse
# import sys
# import glob
# import seaborn as sns

# # 解析命令行参数
# parser = argparse.ArgumentParser()
# parser.add_argument('--runs_dir', default='results_drone_Plz_Converge', help='Directory containing log csv files')
# parser.add_argument('--output_dir', default='analysis_results_Plz_Converge2', help='Directory to save analysis results')

# # 适配 Colab/Jupyter 环境
# if 'ipykernel' in sys.modules:
#     args = parser.parse_args(args=['--runs_dir', 'results_drone_Plz_Converge', '--output_dir', 'analysis_results_Plz_Converge2'])
# else:
#     args = parser.parse_args()

# # 创建输出目录
# os.makedirs(args.output_dir, exist_ok=True)

# # === 修改的核心部分：直接读取目录下所有的 csv 日志文件 ===
# all_data = []
# # 查找所有的 csv 文件
# csv_pattern = os.path.join(args.runs_dir, '*.csv')
# csv_files = glob.glob(csv_pattern)

# for file_path in csv_files:
#     # 排除掉之前生成的 combined 汇总文件，防止重复读取
#     if 'combined' in file_path:
#         continue

#     df = pd.read_csv(file_path)
#     # 用文件名（比如 training_log_xxx.csv）作为这一次运行的 ID
#     df['run_id'] = os.path.basename(file_path)
#     all_data.append(df)

# if not all_data:
#     print(f"错误: 在 {args.runs_dir} 目录下没有找到任何 CSV 日志文件！请检查训练是否成功生成了日志。")
#     sys.exit()

# # 合并所有运行数据
# combined_df = pd.concat(all_data, ignore_index=True)

# # 计算基本统计量
# mean_reward = combined_df.groupby('episode')['reward'].mean().reset_index()
# std_reward = combined_df.groupby('episode')['reward'].std().reset_index()

# # 数据有效性检查
# print(f"调试信息: 找到 {len(all_data)} 个日志文件")
# print(f"调试信息: 唯一episode数量: {len(combined_df['episode'].unique())}")
# print(f"调试信息: 输出目录: {args.output_dir}")

# if len(combined_df['episode'].unique()) < 2:
#     print("警告: 检测到数据中仅包含单个episode，无法生成有效曲线")
#     # 生成单数据点占位图像
#     plt.figure(figsize=(12, 8))
#     episode_value = combined_df['episode'].unique()[0]
#     reward_mean = combined_df['reward'].mean()
#     plt.scatter(episode_value, reward_mean, color='red')
#     plt.xlabel('Episode')
#     plt.ylabel('Reward')
#     plt.title('Insufficient Data: Only Single Episode Available')
#     plt.grid(True)
#     plt.savefig(os.path.join(args.output_dir, 'all_runs_reward_curve.png'))
#     plt.close()
# else:
#     # 绘制所有运行的奖励曲线
#     plt.figure(figsize=(12, 8))
#     sns.lineplot(data=combined_df, x='episode', y='reward', hue='run_id', alpha=0.5)
#     plt.xlabel('Episode')
#     plt.ylabel('Reward')
#     plt.title('Reward Curves for All Training Runs')
#     plt.grid(True)
#     # 把图例移到图外边，防止遮挡曲线
#     plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
#     plt.tight_layout()
#     plt.savefig(os.path.join(args.output_dir, 'all_runs_reward_curve.png'))
#     plt.close()

#     # 绘制平均奖励曲线
#     plt.figure(figsize=(12, 8))
#     plt.plot(mean_reward['episode'], mean_reward['reward'], label='Mean Reward', color='blue')
#     plt.fill_between(mean_reward['episode'],
#                      mean_reward['reward'] - std_reward['reward'],
#                      mean_reward['reward'] + std_reward['reward'],
#                      alpha=0.2, color='blue', label='Std Dev')
#     plt.xlabel('Episode')
#     plt.ylabel('Reward')
#     plt.title('Average Reward with Standard Deviation')
#     plt.legend()
#     plt.grid(True)
#     plt.savefig(os.path.join(args.output_dir, 'average_reward_curve.png'))
#     plt.close()

# # 绘制奖励分布箱线图 (如果 Episode 很多，箱线图会很密，这里做了每 10 个 episode 采样)
# plt.figure(figsize=(12, 8))
# sampled_df = combined_df[combined_df['episode'] % 10 == 0] # 只画出 0, 10, 20... 的箱线图
# if not sampled_df.empty:
#     sns.boxplot(data=sampled_df, x='episode', y='reward')
#     plt.xlabel('Episode (Sampled every 10)')
#     plt.ylabel('Reward Distribution')
#     plt.title('Reward Distribution Across Episodes')
#     plt.xticks(rotation=45)
#     plt.grid(True)
#     plt.tight_layout()
#     plt.savefig(os.path.join(args.output_dir, 'reward_boxplot.png'))
# plt.close()

# # 输出总体统计信息
# print('所有运行的总体统计信息：')
# print(combined_df[['reward', 'steps']].describe())

# # 保存合并后的数据
# combined_df.to_csv(os.path.join(args.output_dir, 'combined_training_log.csv'), index=False)
# print(f"✅ 所有图表和数据已成功保存至: {args.output_dir}")

/tmp/ipykernel_1163/2045113682.py:45: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined_df = pd.concat(all_data, ignore_index=True)


调试信息: 找到 5 个日志文件
调试信息: 唯一episode数量: 5000
调试信息: 输出目录: analysis_results_Plz_Converge2
所有运行的总体统计信息：
            reward
count  5010.000000
mean   -187.981905
std      68.017060
min    -363.811175
25%    -230.559175
50%    -196.022395
75%    -156.474063
max     205.000345
✅ 所有图表和数据已成功保存至: analysis_results_Plz_Converge2


In [ ]:
# # 压缩并下载训练结果文件夹，包含模型和分析结果。注意路径要根据实际情况调整。
# # 1. 使用 zip 命令将整个文件夹压缩为 .zip 文件
# # -r 表示递归压缩（包含文件夹内的所有子文件和子文件夹）
# !zip -r /content/results_drone_Plz_Converge.zip /content/results_drone_Plz_Converge

# !zip -r /content/analysis_results_Plz_Converge.zip /content/analysis_results_Plz_Converge
# # 2. 引入 Colab 的文件下载模块
# from google.colab import files

# # 3. 触发浏览器下载
# files.download('/content/results_drone_Plz_Converge.zip')
# files.download('/content/analysis_results_Plz_Converge.zip')

  adding: content/results_drone_Plz_Converge/ (stored 0%)
  adding: content/results_drone_Plz_Converge/td3_critic_3800.pth (deflated 8%)
  adding: content/results_drone_Plz_Converge/td3_actor_2300.pth (deflated 8%)
  adding: content/results_drone_Plz_Converge/td3_actor_2900.pth (deflated 8%)
  adding: content/results_drone_Plz_Converge/td3_actor_1400.pth (deflated 8%)
  adding: content/results_drone_Plz_Converge/td3_critic_2700.pth (deflated 8%)
  adding: content/results_drone_Plz_Converge/td3_critic_2400.pth (deflated 8%)
  adding: content/results_drone_Plz_Converge/td3_actor_4900.pth (deflated 7%)
  adding: content/results_drone_Plz_Converge/td3_critic_1800.pth (deflated 8%)
  adding: content/results_drone_Plz_Converge/td3_critic_1000.pth (deflated 8%)
  adding: content/results_drone_Plz_Converge/td3_actor_4500.pth (deflated 7%)
  adding: content/results_drone_Plz_Converge/td3_actor_2700.pth (deflated 8%)
  adding: content/results_drone_Plz_Converge/td3_critic_4800.pth (deflated 7%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# 三维仿真前置:导入权重文件
import os
import torch

# 确保设备设置正确
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
directory = "results_drone_Plz_Converge" # 你的存档文件夹

print(f"🔍 正在扫描文件夹 {directory} ...")
if not os.path.exists(directory):
    print("❌ 文件夹不存在，请检查路径！")
else:
    # 1. 寻找所有的 .pth 权重文件
    files = os.listdir(directory)
    pth_files = [f for f in files if f.endswith('.pth')]
    print(f"找到的权重文件: {pth_files}")

    if len(pth_files) == 0:
        print("❌ 没有找到任何 .pth 文件！请检查你的模型是否存到了其他地方。")
    else:
        # 2. 优先寻找名字里带 'actor' 的文件，如果没有就随便拿第一个
        actor_file = next((f for f in pth_files if 'actor' in f.lower()), pth_files[0])
        file_path = os.path.join(directory, actor_file)

        print(f"🚀 准备将记忆注入大脑: {file_path}")

        # 3. 使用 PyTorch 原生方法强行加载权重
        try:
            # 读取文件，并确保它被放到正确的设备(GPU/CPU)上
            state_dict = torch.load(file_path, map_location=device)
            agent.actor.load_state_dict(state_dict)
            print("✅ 成功加载满级大脑！现在可以去跑画 3D 录像的代码了！")

        except RuntimeError as e:
            print(f"❌ 加载失败！网络维度可能不匹配（比如存档是 6 维的，但你现在用的是 14 维的网络）。")
            print(f"详细报错: {e}")
        except Exception as e:
            print(f"❌ 发生未知错误: {e}")

🔍 正在扫描文件夹 results_drone_Plz_Converge ...
找到的权重文件: ['td3_critic_3800.pth', 'td3_actor_2300.pth', 'td3_actor_2900.pth', 'td3_actor_1400.pth', 'td3_critic_2700.pth', 'td3_critic_2400.pth', 'td3_actor_4900.pth', 'td3_critic_1800.pth', 'td3_critic_1000.pth', 'td3_actor_4500.pth', 'td3_actor_2700.pth', 'td3_critic_4800.pth', 'td3_actor_3500.pth', 'td3_actor_1600.pth', 'td3_actor_300.pth', 'td3_critic_600.pth', 'td3_actor_900.pth', 'td3_actor_2100.pth', 'td3_actor_1500.pth', 'td3_critic_3200.pth', 'td3_critic_4300.pth', 'td3_actor_3200.pth', 'td3_critic_4000.pth', 'td3_actor_4600.pth', 'td3_critic_4500.pth', 'td3_critic_2500.pth', 'td3_actor_2400.pth', 'td3_critic_3400.pth', 'td3_critic_1100.pth', 'td3_actor_4700.pth', 'td3_actor_4100.pth', 'td3_actor_3400.pth', 'td3_critic_3500.pth', 'td3_actor_1000.pth', 'td3_actor_3700.pth', 'td3_actor_1700.pth', 'td3_actor_3300.pth', 'td3_actor_3600.pth', 'td3_critic_900.pth', 'td3_actor_500.pth', 'td3_critic_3900.pth', 'td3_critic_1300.pth', 'td3_actor_7

In [ ]:
# 三维仿真: 纯净测试
import numpy as np
import torch

# 1. 临时关闭探索噪声，进行“纯净测试”
# 确保你已经实例化了 env 和 agent，并且加载了最好的模型权重
# original_noise = agent.policy_noise
# agent.policy_noise = 0.0  # 设为0，让它完全按照学到的最优策略飞行

# 2. 初始化环境，准备“黑匣子”记录器
state = env.reset()
done = False

# 记录核心数据
trajectory = []                # 记录飞行的轨迹坐标
start_pos = env.position.copy() # 记录起点
target_pos = env.target.copy()  # 记录终点
obstacles_data = env.obstacles.copy() # 记录障碍物的坐标和半径

# 记录起点
trajectory.append(start_pos.copy())

# 3. 开始闭环飞行测试
print("🚁 无人机起飞，开始纯净飞行测试...")
step_count = 0

while not done:
    # 获取动作 (此时没有随机噪声)
    # action = agent.select_action(state)
    action = agent.select_action(state, exploration_noise=0.0)
    # 执行动作
    state, reward, done, info = env.step(action)

    # 记录当前位置
    trajectory.append(env.position.copy())
    step_count += 1

# 转换为 NumPy 数组方便后续画图
trajectory = np.array(trajectory)

# 恢复训练时的噪声设置（好习惯）
# agent.policy_noise = original_noise

# 打印最终结果
final_distance = np.linalg.norm(env.target - env.position)
print(f"✅ 飞行结束！共耗时 {step_count} 步。")
if final_distance < 1.5:
    print(f"🎯 成功到达终点！距目标仅 {final_distance:.2f} 米。")
else:
    print(f"💥 发生碰撞或超时。距目标还有 {final_distance:.2f} 米。")

🚁 无人机起飞，开始纯净飞行测试...
✅ 飞行结束！共耗时 41 步。
💥 发生碰撞或超时。距目标还有 24.72 米。


# 🎯 三维仿真依赖包👇记得下！

In [ ]:
!pip install plotly
!pip install --upgrade nbformat

In [ ]:
#三维仿真绘图
import plotly.graph_objects as go
import numpy as np

# 1. 初始化 3D 画布
fig = go.Figure()

# 2. 画起点和终点
fig.add_trace(go.Scatter3d(
    x=[start_pos[0]], y=[start_pos[1]], z=[start_pos[2]],
    mode='markers', marker=dict(size=6, color='blue'), name='起点 (Start)'
))
fig.add_trace(go.Scatter3d(
    x=[target_pos[0]], y=[target_pos[1]], z=[target_pos[2]],
    mode='markers', marker=dict(size=8, color='green', symbol='diamond'), name='终点 (Target)'
))

# 3. 画无人机的飞行轨迹线
fig.add_trace(go.Scatter3d(
    x=trajectory[:, 0], y=trajectory[:, 1], z=trajectory[:, 2],
    mode='lines+markers',
    line=dict(color='orange', width=6),
    marker=dict(size=3, color='orange'),
    name='无人机轨迹'
))

# 4. 数学建模：生成 3D 球体表面的网格数据
def create_sphere_mesh(center, radius, resolution=20):
    u = np.linspace(0, 2 * np.pi, resolution)
    v = np.linspace(0, np.pi, resolution)
    x = center[0] + radius * np.outer(np.cos(u), np.sin(v))
    y = center[1] + radius * np.outer(np.sin(u), np.sin(v))
    z = center[2] + radius * np.outer(np.ones(np.size(u)), np.cos(v))
    return x, y, z

# 5. 把所有障碍物画到图中
for i, obs in enumerate(obstacles_data):
    cx, cy, cz = obs['pos']
    r = obs['radius']
    sx, sy, sz = create_sphere_mesh((cx, cy, cz), r)

    fig.add_trace(go.Surface(
        x=sx, y=sy, z=sz,
        colorscale='Reds',      # 红色代表危险
        opacity=0.4,            # 半透明，避免挡住视线
        showscale=False,
        name=f'障碍物 {i+1}'
    ))

# 6. 设置画布的视角和比例
fig.update_layout(
    scene=dict(
        xaxis_title='X (米)',
        yaxis_title='Y (米)',
        zaxis_title='Z (米)',
        # 【关键设置】 aspectmode='data' 强制 XYZ 比例为 1:1:1，确保球体不会变成椭圆
        aspectmode='data'
    ),
    title="无人机 3D 避障航线分析视图",
    margin=dict(l=0, r=0, b=0, t=40),
    legend=dict(x=0.02, y=0.98)
)

# 7. 显示交互图表
fig.show()
# fig.show(renderer="notebook")#适合大多数网页版 Jupyter
# fig.show(renderer="iframe")#适合经典的 Jupyter Notebook
# fig.show(renderer="vscode")#VS Code 的网页版
fig.write_html("drone_3d_flight_Plz_Converge.html")#直接导出为独立 HTML 网页
print("3D仿真图已保存为 drone_3d_flight_Plz_Converge.html！")

3D仿真图已保存为 drone_3d_flight_Plz_Converge.html！


In [ ]:
#生成 3D 飞行录像代码
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML
# 导入 3D 绘图模块
from mpl_toolkits.mplot3d import Axes3D

# 1. 初始化画布和 3D 坐标轴
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# 2. 画出起点（蓝色）和终点（绿色星号）
ax.scatter(*start_pos, color='blue', s=100, label='Start', edgecolors='black')
ax.scatter(*target_pos, color='green', s=150, marker='*', label='Target', edgecolors='black')

# 3. 绘制所有的障碍物（红色半透明网格球体）
for obs in obstacles_data:
    cx, cy, cz = obs['pos']
    r = obs['radius']
    u = np.linspace(0, 2 * np.pi, 20)
    v = np.linspace(0, np.pi, 20)
    x = cx + r * np.outer(np.cos(u), np.sin(v))
    y = cy + r * np.outer(np.sin(u), np.sin(v))
    z = cz + r * np.outer(np.ones(np.size(u)), np.cos(v))
    # 使用 wireframe 画网格，避免遮挡视线
    ax.plot_wireframe(x, y, z, color='red', alpha=0.2)

# 4. 固定坐标轴的显示范围
# 之前你的代码里撞墙边界是 15，所以我们把视野固定在 [-15, 15]
# 这样可以防止动画播放时镜头跟着无人机乱晃
ax.set_xlim([-15, 15])
ax.set_ylim([-15, 15])
ax.set_zlim([-15, 15])
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title('Drone 3D Flight Animation')
ax.legend()

# 5. 初始化动画中的“动态元素”：飞行轨迹线 和 无人机当前位置点
trail_line, = ax.plot([], [], [], color='orange', linewidth=2, label='Trajectory')
drone_point, = ax.plot([], [], [], 'o', color='red', markersize=8, label='Drone')

# 动画初始化函数
def init():
    trail_line.set_data([], [])
    trail_line.set_3d_properties([])
    drone_point.set_data([], [])
    drone_point.set_3d_properties([])
    return trail_line, drone_point

# 动画的每一帧更新函数
def update(frame):
    # 获取从第 0 帧到当前帧的所有历史轨迹
    current_traj = trajectory[:frame+1]

    # 更新轨迹线
    trail_line.set_data(current_traj[:, 0], current_traj[:, 1])
    trail_line.set_3d_properties(current_traj[:, 2])

    # 更新无人机当前的红点位置
    current_pos = trajectory[frame]
    drone_point.set_data([current_pos[0]], [current_pos[1]])
    drone_point.set_3d_properties([current_pos[2]])

    return trail_line, drone_point

print("⏳ 正在全力渲染 3D 逐帧动画，请稍候（可能需要几十秒）...")

# 6. 生成动画！(interval=50 表示每帧间隔 50 毫秒，即 20 FPS)
# frames 取轨迹的总步数
ani = animation.FuncAnimation(fig, update, frames=len(trajectory),
                              init_func=init, blit=False, interval=50)

# 7. 关闭静态多余的图表显示，并将动画转为 HTML5 交互式播放器
plt.close()
HTML(ani.to_jshtml())

⏳ 正在全力渲染 3D 逐帧动画，请稍候（可能需要几十秒）...
